# Занятие 3. Фильтрация данных и обработка пропусков

> **Цель занятия:** научиться отбирать нужные строки из таблицы по условиям
> и справляться с «дырками» в данных.

**Что будет:**
1. **Булево маскирование** — как отбирать строки по условию;
2. **Сложные условия** — объединяем через **И (`&`)**, **ИЛИ (`|`)**, **НЕ (`~`)**;
3. **`.query()`** — более читаемый способ записи условий;
4. **`.copy()`** — зачем копировать таблицу перед изменениями;
5. **Пропущенные значения** — как их находить (`.isna()`), удалять (`.dropna()`)
   и заполнять (`.fillna()`).

Все примеры разбираем на **датасете** из прошлого занятия — таблице с результатами ЕГЭ.

In [ ]:
import numpy as np
import pandas as pd

## Немного об удобстве: настройки отображения Pandas

По умолчанию Pandas старается **не перегружать экран**: если столбцов много,
часть из них он прячет и заменяет многоточием `...`. То же самое со строками и
широкими текстами — они обрезаются. Для маленьких табличек это удобно, но когда
мы работаем с реальными данными (у нас **16 столбцов**), хочется **видеть
всё сразу**.

Управлять этим помогает функция **`pd.set_option()`**. Она меняет глобальные
настройки отображения — как таблица будет **печататься** на экран.
Сами данные при этом не меняются, меняется только их «вид».

**Самые полезные настройки:**

| Настройка | Что делает | Пример |
|---|---|---|
| `display.max_columns` | сколько столбцов показывать целиком | `pd.set_option('display.max_columns', 20)` |
| `display.max_rows` | сколько строк показывать | `pd.set_option('display.max_rows', 100)` |
| `display.width` | ширина вывода в символах | `pd.set_option('display.width', 200)` |
| `display.max_colwidth` | максимальная ширина одной ячейки | `pd.set_option('display.max_colwidth', 100)` |
| `display.precision` | сколько знаков после запятой у чисел | `pd.set_option('display.precision', 2)` |

**Как это работает:**
- значение `20` — «показывай до 20 столбцов, не прячь»;
- значение `None` — «не ограничивай вообще, показывай **всё**»;
- вернуть настройку по умолчанию можно через `pd.reset_option('display.max_columns')`.

> **Совет:** в начале ноутбука полезно один раз задать удобные настройки —
> дальше все `df.head()` и подобные вызовы будут показывать данные так, как вам
> нравится, без сюрпризов с многоточиями.

Поставим `max_columns = 20`, чтобы все 16 столбцов таблицы влезли на экран целиком.

In [ ]:
pd.set_option('display.max_columns', 20)      # показывать до 20 столбцов
pd.set_option('display.width', 200)           # шире область вывода
pd.set_option('display.precision', 2)         # 2 знака после запятой у чисел

## Загружаем данные про ЕГЭ

Файл `ege_students.csv`. В нём **1000 учеников** и **16 столбцов**:
имя, пол, возраст, город, школа, любимый предмет, часы подготовки, репетитор, пробники,
пропуски уроков, мотивация, часы сна, средний балл года, балл ЕГЭ и признак «сдал».

In [ ]:
df = pd.read_csv('data/ege_students.csv', sep=';')
print('Размер таблицы:', df.shape)
df.head()

In [ ]:
df.dtypes

---
## Часть 1. Булево маскирование — отбираем строки по условию

Часто нужно выбрать не всю таблицу, а только её **часть**: «только девочки»,
«только те, кто сдал», «только с баллом выше 80». Для этого в Pandas есть
специальный приём — **булево маскирование** (или «логическое индексирование»).

**Идея простая, в два шага:**
1. Пишем условие про столбец — получаем **маску**: колонку из значений `True` / `False`,
   по одному на каждую строку таблицы.
2. Подставляем эту маску в квадратные скобки `df[...]` — Pandas **оставит только те строки,
   где стоит `True`**.

Сначала посмотрим, как выглядит маска.

In [ ]:
# условие: балл ЕГЭ выше 80
mask = df['балл_егэ'] > 80
mask.head(10)

**Вопрос:** мы получили колонку из `True`/`False`. Как теперь с её помощью
отобрать из таблицы именно те строки, где условие выполняется?

Теперь применим маску к таблице.

In [ ]:
high_score = df[df['балл_егэ'] > 80]
print('Всего строк:', len(df))
print('С баллом > 80:', len(high_score))
high_score.head()

**Вопрос:** в квадратных скобках `df[...]` мы обычно писали имя столбца.
А здесь оказалось целое условие. Почему такая запись вообще работает?

### Простые примеры условий

Так же можно фильтровать и по текстовым столбцам — через `==` (равно):

In [ ]:
# только ученики из Москвы
df[df['город'] == 'Москва'].head()

In [ ]:
# только те, кто НЕ сдал

df[...].head()

### Что за операторы можно использовать?

| Оператор | Смысл | Пример |
|---|---|---|
| `==` | равно | `df['пол'] == 'Ж'` |
| `!=` | не равно | `df['город'] != 'Москва'` |
| `>` `>=` | больше / больше-равно | `df['балл_егэ'] >= 80` |
| `<` `<=` | меньше / меньше-равно | `df['возраст'] < 18` |
| `.isin([...])` | значение из списка | `df['город'].isin(['Москва', 'Казань'])` |

Про `.isin()` отдельно: удобно, когда нужно проверить, попадает ли значение в **список**
допустимых. Иначе пришлось бы писать длинную цепочку `или`.

In [ ]:
# ученики из двух столиц
cities = ['Москва', 'Санкт-Петербург']
df[df['город'].isin(cities)].head()

---
## Часть 2. Сложные условия: И, ИЛИ, НЕ

Одного условия часто мало. Например: «девочки **И** сдали», «Москва **ИЛИ** Питер»,
«**НЕ** из Москвы». Такие сложные условия собирают из простых через **логические операторы**.

**Вопрос:** в обычном Python для «И» и «ИЛИ» есть `and` и `or`. Почему в Pandas
для тех же самых логических связок используют другие символы — `&`, `|`, `~`?

> **Важное правило:** каждое условие **оборачиваем в круглые скобки**.
> Без скобок Pandas запутается в порядке операций и выдаст ошибку.

### Оператор И (`&`) — оба условия одновременно

In [ ]:
# девочки И сдали
mask = (df['пол'] == 'Ж') & (df['сдал'] == 'да')
df[mask].head()

In [ ]:
# балл ЕГЭ выше 80 И больше 10 часов подготовки в неделю
mask = (...)
df[mask].head()

### Оператор ИЛИ (`|`) — хотя бы одно из условий

In [ ]:
# из Москвы ИЛИ из Санкт-Петербурга
mask = (df['город'] == 'Москва') | (df['город'] == 'Санкт-Петербург')
df[mask].head()

### Оператор НЕ (`~`) — переворачивает True в False

In [ ]:
# НЕ из Москвы
mask = ~(df['город'] == 'Москва')
df[mask].head()

### Комбинируем несколько условий

Условий может быть сколько угодно — главное каждое **в скобках** и правильный оператор между ними.

In [ ]:
# девочки-лицеистки, которые сдали на 70+
mask = (df['пол'] == 'Ж') & (df['тип_школы'] == 'Лицей') & (df['балл_егэ'] >= 70)
df[mask].head()

> **Частая ошибка:** написать `and` вместо `&` или забыть скобки.
> Тогда Python выдаёт ошибку про «truth value of a Series is ambiguous».
> Если увидите её — проверьте: везде ли `&` `|` `~` и все ли условия в круглых скобках.

---
## Часть 3. Метод `.query()` — то же самое, но читаемее

Когда условий много, запись со скобками и амперсандами становится громоздкой.
У Pandas есть **альтернатива** — метод **`.query()`**. В нём условие пишут **строкой**,
почти как обычным языком.

Сравните две записи одного и того же:

In [ ]:
# через булеву маску
df[(df['пол'] == 'Ж') & (df['балл_егэ'] >= 70)].head()

In [ ]:
# то же самое через .query()
df.query('пол == "Ж" and балл_егэ >= 70').head()

**Что удобно в `.query()`:**
- пишем **`and` / `or` / `not`** — как в обычном Python;
- **не нужны** ни квадратные скобки `df[...]`, ни лишние `df['...']` вокруг имён столбцов;
- условие читается почти как фраза на языке.

**Синтаксис:**
- имена столбцов пишем **прямо** (без кавычек и без `df[...]`);
- **строковые значения** — в кавычках: `'да'` или `"да"`;
- числа — как есть.

### Ещё примеры

In [ ]:
# балл ЕГЭ выше 80 и больше 10 часов подготовки
df.query('балл_егэ > 80 and часов_подготовки_в_неделю > 10').head()

In [ ]:
# из Москвы или Питера
df.query('...').head()

In [ ]:
# то же через in — короче
df.query('город in ["Москва", "Санкт-Петербург"]').head()

In [ ]:
# НЕ из Москвы
df.query('...').head()

**Вопрос:** обе записи возвращают одну и ту же таблицу. В каких случаях удобнее
булева маска, а в каких — `.query()`?

---
## Часть 4. `.copy()` — зачем копировать таблицу

Когда мы отбираем часть строк — `df_high = df[df['балл_егэ'] > 80]` — Pandas часто
возвращает **не самостоятельную новую таблицу, а «взгляд»** (view) на кусок исходной.

Если потом попробовать что-то поменять в этом кусочке, Pandas ругается предупреждением
`SettingWithCopyWarning` — потому что непонятно, куда именно применять изменения:
в кусочек или в исходную таблицу.

**Решение простое:** если вы собираетесь **менять** отфильтрованные данные — сразу вызывайте
`.copy()`. Это делает **самостоятельную копию**, с которой можно работать безопасно.

In [ ]:
# делаем самостоятельную копию — с ней можно работать спокойно
high_score = df[df['балл_егэ'] > 80].copy()
high_score['отлично'] = 'да'   # добавили столбец в копию
high_score.head()

In [ ]:
# в исходной таблице никакого столбца 'отлично' нет — copy сработал
print('Столбцы df:', list(df.columns))

**Правило простое:**
> Если после фильтрации будете **что-то менять** — вызывайте `.copy()`.
> Если только смотрите или считаете — можно без него.

---
## Часть 5. Пропущенные значения (NaN)

В реальных данных часто чего-то **не хватает**: ученик забыл заполнить анкету, датчик
сломался, поле оставили пустым. Такие «дырки» в Pandas обозначаются специальным значением
**`NaN`** (от англ. *Not a Number* — «не число»).

Работать с NaN нужно осторожно: обычные вычисления с ними дают снова NaN
(`5 + NaN = NaN`), а сравнения возвращают `False`.

### Шаг 1. Находим пропуски — `.isna()`

Метод **`.isna()`** возвращает такую же таблицу из `True` / `False`: `True` — где пропуск.
Ещё есть `.notna()` — наоборот, `True` где значение есть.

In [ ]:
# сколько пропусков в каждом столбце
df.isna().sum()

**Вопрос:** мы первым делом посмотрели, где и сколько пропусков.
Почему это важно сделать до того, как что-то удалять или заполнять?

In [ ]:
# посмотрим на строки, где нет мотивации
df[df['мотивация_1_10'].isna()]

### Шаг 2. Удаляем строки с пропусками — `.dropna()`

Самый простой способ — **выкинуть** строки, в которых есть пропуски. Метод
**`.dropna()`** удаляет все строки, где хотя бы в одном столбце стоит `NaN`.

In [ ]:
df_clean = df.dropna()
print('Было строк:', len(df))
print('Осталось после dropna:', len(df_clean))
print('Удалили:', len(df) - len(df_clean), 'строк с пропусками')

Можно удалять пропуски **только по одному столбцу**, если остальные не так важны:

In [ ]:
# выбрасываем только те строки, где нет часов сна
df_sleep = df.dropna(subset=['часов_сна'])
print('Осталось:', len(df_sleep))

> **Осторожно:** `.dropna()` может выкинуть очень много данных, если пропусков много.
> В нашем случае — всего ~65 строк из 1000, потеря небольшая. Но если пропусков 30–50%,
> лучше их **заполнить**, а не удалять.

### Шаг 3. Заполняем пропуски — `.fillna()`

Метод **`.fillna(значение)`** ставит на место каждого `NaN` то, что мы укажем.
Часто используют:
- **среднее** значение по столбцу — для числовых данных;
- **медиану** — если есть выбросы;
- **`0`** или специальное значение — если это осмысленно;
- **самое частое значение** — для категорий.

In [ ]:
# заполним пропуски в часах сна средним значением
avg_sleep = df['часов_сна'].mean()
print('Среднее число часов сна:', round(avg_sleep, 2))

df_filled = df.copy()
df_filled['часов_сна'] = df_filled['часов_сна'].fillna(avg_sleep)
print('Пропусков в часах сна теперь:', df_filled['часов_сна'].isna().sum())

In [ ]:
# мотивацию заполним медианой (это оценка от 1 до 10 — целое число)
med_mot = df['мотивация_1_10'].median()
print('Медиана мотивации:', med_mot)

df_filled['мотивация_1_10'] = df_filled['мотивация_1_10'].fillna(med_mot)
print('Пропусков теперь:', df_filled['мотивация_1_10'].isna().sum())

In [ ]:
# проверим: пропусков больше нет
df_filled.isna().sum()

### Что выбрать: удалять или заполнять?

| Ситуация | Что делать |
|---|---|
| Пропусков **очень мало** (< 5%) | Проще **удалить** через `.dropna()` |
| Пропусков много, столбец важный | **Заполнить** через `.fillna()` (среднее / медиана / мода) |
| Столбец **почти пустой** | Удалить весь столбец |
| Пропуск — это **осмысленный «ноль»** | Заполнить нулём: `.fillna(0)` |

**Главное правило:** прежде чем что-то делать с пропусками — **посмотрите, где они и сколько их**.
Обычно это первый шаг любого анализа.

---
## Итог занятия

Сегодня научились:
- **отбирать строки** по условию через **булеву маску** `df[df['столбец'] > ...]`;
- собирать **сложные условия** через `&` (И), `|` (ИЛИ), `~` (НЕ) — не забывая скобки;
- писать те же условия **читаемо** через **`.query()`**;
- копировать таблицу через **`.copy()`**, если собираемся её менять;
- находить пропуски через **`.isna()`**, удалять их через **`.dropna()`**
  и заполнять через **`.fillna()`**.

**Главная мысль:** реальные данные почти никогда не бывают идеальными.
Уметь **быстро отобрать нужный кусок** и **аккуратно обработать пропуски** — базовый
навык любого анализа. Без него все дальнейшие расчёты будут неверными.